In [ ]:
# 3.02 — Baseline: Regressão Logística
#
# Objetivo: treinar LogisticRegression como baseline linear, comparar com o
# DummyClassifier e registrar métricas, artefatos e dataset no MLflow.
#
# Decisões técnicas:
# - class_weight="balanced" — test set tem desbalanceamento residual (~26% churn)
# - solver="lbfgs" — eficiente para datasets de tamanho médio (~5 k amostras)
# - max_iter=1000 — garante convergência sem warning
# - Validação cruzada estratificada (k=5) sobre X_train (pós-SMOTE+ENN)
# - X_test intocado — avaliação final única, nunca usado no desenvolvimento

import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

# ── Raiz do projeto no path ───────────────────────────────────────────────────
sys.path.insert(0, str(Path("..").resolve()))

from churn_telecom.config import (
    DATA_PROCESSED,
    RANDOM_STATE,
    REPORTS_FIGURES,
    get_logger,
    log_dataset_to_mlflow,
    mlflow_run,
    setup_mlflow,
)
from churn_telecom.plots import (
    save_confusion_matrix,
    save_feature_importance,
    save_roc_curve,
)


In [ ]:
logger = get_logger("3.02_baseline_logistic")
logger.info("Iniciando 3.02_vab_baseline_logistic")


In [ ]:
# Carrega train/test processados pelo pipeline do notebook 1.03.
# - train.parquet : 5395 amostras, 30 features (pós-SMOTE+ENN, churn ~57%)
# - test.parquet  : 1405 amostras, 30 features (intocado,  churn ~26%)

TRAIN_PATH = DATA_PROCESSED / "train.parquet"
TEST_PATH = DATA_PROCESSED / "test.parquet"

assert TRAIN_PATH.exists(), f"Arquivo não encontrado: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Arquivo não encontrado: {TEST_PATH}"

train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

TARGET = "churn_value"
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

feature_names: list[str] = X_train.columns.tolist()

logger.info("Train : %s | churn: %.2f%%", X_train.shape, y_train.mean() * 100)
logger.info("Test  : %s | churn: %.2f%%", X_test.shape, y_test.mean() * 100)
logger.info("Features: %d", len(feature_names))


In [ ]:
# Fluxo dentro do run:
# 1. Tags descritivas (notebook, fase, modelo, framework)
# 2. Dataset versionado via hash MD5 (train + test)
# 3. Parâmetros do modelo
# 4. Validação cruzada estratificada (k=5)
# 5. Treino final no X_train completo
# 6. Predições no X_test (uso único)
# 7. Métricas: AUC-ROC, PR-AUC, F1, Recall, Precision + desvios do CV
# 8. Artefatos: confusion matrix, ROC curve, feature importance
# 9. Coeficientes como JSON
# 10. Registro no Model Registry

setup_mlflow()

MODEL_NAME = "baseline-logistic"
FIGURES_DIR = REPORTS_FIGURES / "baselines"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with mlflow_run("3.02_baseline_logistic") as run:

    # ── 1. Tags ───────────────────────────────────────────────────────────────
    mlflow.set_tags(
        {
            "notebook": "3.02",
            "fase": "baseline",
            "modelo": "logistic",
            "task": "classification",
            "framework": "sklearn",
        }
    )

    # ── 2. Dataset versionado ─────────────────────────────────────────────────
    log_dataset_to_mlflow(X_train, y_train, split="train", source_path=TRAIN_PATH)
    log_dataset_to_mlflow(X_test, y_test, split="test", source_path=TEST_PATH)

    # ── 3. Parâmetros ─────────────────────────────────────────────────────────
    params = {
        "class_weight": "balanced",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "solver": "lbfgs",
        "C": 1.0,
        "cv_folds": 5,
    }
    mlflow.log_params(params)
    logger.info("Params: %s", params)

    # ── 4. Validação cruzada estratificada ────────────────────────────────────
    model = LogisticRegression(
        class_weight=params["class_weight"],
        max_iter=params["max_iter"],
        random_state=params["random_state"],
        solver=params["solver"],
        C=params["C"],
    )
    cv = StratifiedKFold(
        n_splits=params["cv_folds"],
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=["average_precision", "roc_auc", "f1", "recall", "precision"],
    )
    logger.info(
        "CV AUC   : %.4f ± %.4f",
        cv_results["test_roc_auc"].mean(),
        cv_results["test_roc_auc"].std(),
    )
    logger.info(
        "CV F1    : %.4f ± %.4f",
        cv_results["test_f1"].mean(),
        cv_results["test_f1"].std(),
    )
    logger.info(
        "CV Recall: %.4f ± %.4f",
        cv_results["test_recall"].mean(),
        cv_results["test_recall"].std(),
    )

    # ── 5. Treino final + predições no test set ───────────────────────────────
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # ── 6. Métricas ───────────────────────────────────────────────────────────
    metrics = {
        # Test set — avaliação final
        "test_auc": roc_auc_score(y_test, y_proba),
        "test_pr_auc": average_precision_score(y_test, y_proba),
        "test_f1": f1_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        # Cross-val — estabilidade do modelo
        "cv_auc_mean": float(cv_results["test_roc_auc"].mean()),
        "cv_auc_std": float(cv_results["test_roc_auc"].std()),
        "cv_pr_auc_mean": float(cv_results["test_average_precision"].mean()),
        "cv_pr_auc_std": float(cv_results["test_average_precision"].std()),
        "cv_f1_mean": float(cv_results["test_f1"].mean()),
        "cv_f1_std": float(cv_results["test_f1"].std()),
        "cv_recall_mean": float(cv_results["test_recall"].mean()),
        "cv_recall_std": float(cv_results["test_recall"].std()),
    }
    mlflow.log_metrics(metrics)
    logger.info("Metrics: %s", {k: f"{v:.4f}" for k, v in metrics.items()})

    # ── 7. Artefatos visuais → reports/figures/baselines/ ────────────────────
    # Salva localmente primeiro (commitado no git), depois loga no MLflow
    mlflow.log_artifact(
        str(
            save_confusion_matrix(
                y_test,
                y_pred,
                FIGURES_DIR / "logistic_confusion_matrix.png",
                title="Confusion Matrix — Logistic Regression",
            )
        ),
        artifact_path="baselines/logistic",
    )

    mlflow.log_artifact(
        str(
            save_roc_curve(
                y_test,
                y_proba,
                FIGURES_DIR / "logistic_roc_curve.png",
                model_name="Logistic Regression",
                title="ROC Curve — Logistic Regression",
            )
        ),
        artifact_path="baselines/logistic",
    )

    coef_abs = np.abs(model.coef_.ravel())
    mlflow.log_artifact(
        str(
            save_feature_importance(
                importances=coef_abs,
                feature_names=feature_names,
                path=FIGURES_DIR / "logistic_feature_importance.png",
                title="Feature Importance — |Coeficientes| Logistic Regression",
                color="steelblue",
                top_n=20,
            )
        ),
        artifact_path="baselines/logistic",
    )

    # ── 8. Coeficientes como JSON ─────────────────────────────────────────────
    coef_dict = {
        name: float(round(coef, 6))
        for name, coef in zip(feature_names, model.coef_.ravel())
    }
    mlflow.log_dict(coef_dict, "baselines/logistic/coefficients.json")

    top5 = sorted(coef_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
    logger.info("Top 5 features por |coef|:")
    for feat, val in top5:
        logger.info("  %-35s : %+.4f", feat, val)

    # ── 9. Registro no Model Registry ─────────────────────────────────────────
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        input_example=X_test.iloc[:5],
    )

    logger.info("Model URI : %s", model_info.model_uri)
    logger.info("Run ID    : %s", run.info.run_id)


In [ ]:
# Análise exploratória — não afeta o registro no MLflow.

coef_series = pd.Series(model.coef_.ravel(), index=feature_names).reindex(
    pd.Series(model.coef_.ravel(), index=feature_names)
    .abs()
    .sort_values(ascending=False)
    .index
)

logger.info(
    "\nTop 10 coeficientes (magnitude):\n%s",
    coef_series.head(10).to_string(),
)
